# Step 2: Filter notebook

Ce notebook filtre les attributs et uniformise le format afin qu'ils soient prêt à être intégré à l'indice. Les filtres sont évalués au cas par cas selon la pertinence.

In [1]:
import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point
import rasterio
import numpy as np
import os
import folium
from folium import Choropleth, CircleMarker, GeoJson
import branca.colormap as cm
from IPython.display import display
pd.set_option('display.max_columns', None)

from feature_to_network import *

## Imports

#### Import des segments

In [2]:
# operation_crs = "EPSG:2056"  # Swiss coordinate system
# target_crs = "EPSG:4326"  # WGS84 coordinate system

# print("Set parameters : GG or GE, bike or walk")
# territory = 'GG' # GG or GE or GGminGE
# network = "walk"  # walk or bike

# if territory == 'GG':
#     input_file_path = '../../Data/input'
#     output_step1_path='../../Data/output/GG/step-1'
#     output_step2_path='../../Data/output/GG/step-2'
#     output_step3_path='../../Data/output/GG/step-3'


#     save_path = '../../Data/output/GG/step-2'

#     attribute_source_path = 'source_path_GG'
#     file_name = 'file_name_GG'  # column name in attributs_info excel

# if territory == 'GE':
#     input_file_path = '../../Data/input'
#     output_step1_path='../../Data/output/GE/step-1'
#     output_step2_path='../../Data/output/GE/step-2'
#     output_step3_path='../../Data/output/GE/step-3'


#     save_path = '../../Data/output/GE/step-2'

#     attribute_source_path = 'source_path_GE'
#     file_name = 'file_name_GE'  # column name in attributs_info excel


# save_filtered_attributes = True

# # Load segments GeoDataFrame (with 'segment_id')
# print("Loading segments...")
# # reLoad pedestrian segments
# segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_all_segments.parquet"))
# segmented_net["geometry"] = segmented_net["geometry"].apply(lambda geom: geom.buffer(0) if not geom.is_valid else geom)
# segmented_net = segmented_net.to_crs(operation_crs)





operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

print("Set parameters : GG or GE, bike or walk")
territory = 'GG' # GG or GE
network = "walk"  # walk ONLY in the notebook, go to 2_1_Filter_features_bike.ipynb for bike

input_file_path = '../../Data/input'
output_step1_path=f"../../Data/output/{network}/{territory}/step-1"
output_step2_path=f"../../Data/output/{network}/{territory}/step-2"
output_step3_path=f"../../Data/output/{network}/{territory}/step-3"
save_path = f"../../Data/output/{network}/{territory}/step-2"
attribute_source_path = f"source_path_{territory}"
file_name = f"file_name_{territory}"  # column name in attributs_info excel

save_filtered_attributes = True
include = f"include_in_index_{territory}"


# Load segments GeoDataFrame (with 'segment_id')
print("Loading segments...")
# reLoad walk segments
segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_all_segments.parquet"))
segmented_net["geometry"] = segmented_net["geometry"].apply(lambda geom: geom.buffer(0) if not geom.is_valid else geom)
segmented_net = segmented_net.to_crs(operation_crs)


Set parameters : GG or GE, bike or walk
Loading segments...


In [3]:
import geopandas as gpd
from shapely.geometry.base import BaseGeometry

# Clean geometries safely
def clean_any_geom(geom):
    if geom is None:
        return None
    if not isinstance(geom, BaseGeometry):
        return None
    if geom.is_valid:
        return geom
    try:
        fixed = geom.buffer(0)
        if fixed is None or fixed.is_empty:
            return geom
        return fixed
    except Exception:
        return geom
        

# Define a function to save the filtered data
def save(save_filtered_attributes, row, gdf, attribute):

    if not save_filtered_attributes:
        print("Note : Save option is disabled.")
        return

    # Parse save formats: comma-separated list allowed
    raw = str(row.get('save_format', '') or '')
    formats = [f.strip().lower() for f in raw.split(',') if f.strip()]
    if not formats:
        formats = ['csv']  # default fallback

    for fmt in formats:
        try:
            if fmt == 'parquet':
                dirpath = f'{output_step2_path}/parquet_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_parquet(f"{dirpath}/{attribute}.parquet")
                print(f"Filtered data saved for attribute: {attribute} in format: parquet")

            elif fmt == 'gpkg' or fmt == 'geopackage':
                dirpath = f'{output_step2_path}/gpkg_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_file(f"{dirpath}/{attribute}.gpkg", driver="GPKG")
                print(f"Filtered data saved for attribute: {attribute} in format: gpkg")

            elif fmt == 'csv':
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                # to_csv may not preserve geometry consistently; keep original behavior
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Filtered data saved for attribute: {attribute} in format: csv")

            else:
                # Unknown format -> fallback to csv and warn
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Warning: Unknown save format '{fmt}' for attribute {attribute}. Data saved as csv.")
        except Exception as e:
            print(f"Error saving {attribute} as {fmt}: {e}")

#### Import des attributs 

In [4]:
attributs_info = pd.read_excel(f"{input_file_path}/attributs/attributs_info_{network}_urbain.xlsx", sheet_name="attributs_info")

# Keep the full attribute table available for per-attribute cells.
# Excel may read include_in_index_* either as booleans or strings depending on the column.
attributs_info["include_in_current_index"] = (
    attributs_info[include]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)
attributs_info["attribute_available_for_territory"] = ~(
    attributs_info[attribute_source_path]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["nan", "none", "", "unavailable"])
)

print(f"Attributes included in index for {territory}: {attributs_info['include_in_current_index'].sum()} / {len(attributs_info)}")
print(f"Attributes available for {territory}: {attributs_info['attribute_available_for_territory'].sum()} / {len(attributs_info)}")

attributs_info

Attributes included in index for GG: 22 / 26
Attributes available for GG: 21 / 26


,Class,meta_attribute,attribute,include_in_index_GG,include_in_index_GE,source_type_GE,source_path_GE,file_name_GE,source_type_VD,source_path_VD,file_name_VD,source_type_FR,source_path_FR,file_name_FR,source_type_GG,source_path_GG,file_name_GG,merge_rule_GG,initial_weight,class_weight,buffer_size_GG,buffer_size_GE,impact_attribut,geometry_type_GE,how_GE,value_column_GE,geometry_type_GG,how_GG,value_column_GG,clip_GG,clip_GE,missing_value_strategy,filter_column,filter_values,crs,save_format,Commentaire,include_in_current_index,attribute_available_for_territory
0,Attractivité,lac_cours_deau,lac_cours_deau,True,True,sitg,attributs/GE/lac_cours_deau,CAD_NATURE_SOL-SHP/CAD_NATURE_SOL.shp,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.25,0.5,50,50,favorable,polygon,presence,NaN,polygon,presence,NaN,NaN,NaN,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",ok,True,True
1,Attractivité,fontaine,fontaine,True,True,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.25,0.5,50,50,favorable,point,count,NaN,point,count,NaN,2.0,2.0,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",ok,True,True
2,Attractivité,espaces_ouverts,espaces_ouverts,True,True,sitg,attributs/GE/espaces_ouverts,OBS_EQUIPEMENTS_ESPACES_PUB-SHP/OBS_EQUIPEMENT...,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.25,0.5,1,1,favorable,polygon,presence,NaN,polygon,presence,NaN,NaN,2.0,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",TB,True,True
3,Attractivité,amenite,rez_actif,True,True,sitg,attributs/GE/rez_actif,REG_ENTREPRISE_ETABLISSEMENT-SHP/REG_ENTREPRIS...,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.50,0.5,50,50,favorable,point,count,NaN,point,count,NaN,4.0,10.0,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",TB,True,True
4,Attractivité,transport_public,tp,True,True,sitg,attributs/GE/tp,TPG_ARRETS-SHP/TPG_ARRETS.shp,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.50,0.5,150,150,favorable,point,presence,NaN,point,presence,NaN,NaN,NaN,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",TB,True,True
5,Attractivité,amenite,amenite,True,True,sitg,attributs/GE/amenite,REG_ENTREPRISE_ETABLISSEMENT-SHP/REG_ENTREPRIS...,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.50,0.5,75,75,favorable,point,count,NaN,point,count,NaN,2.0,6.0,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",TB,True,True
6,Commodité,attente,attente,False,False,NaN,attributs/GE/attente,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.5,10,10,defavorable,point,sum,completer,point,sum,completer,NaN,NaN,keep_nan,filtered,1,2056,"parquet, csv, gpkg",NaN,False,False
7,Commodité,bruit,bruit,Unavailable,True,sitg,attributs/GE/bruit,SPBR_SECTEUR_EXPOSE_AU_BRUIT_2025/SPBR_SECTEUR...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unavailable,NaN,NaN,0.00,0.5,1,1,defavorable,polygon,presence,NaN,polygon,presence,NaN,NaN,NaN,mean_if_missing,filtered,1,2056,"parquet, csv, gpkg",NaN,False,False
8,Commodité,temperature,temperature,True,True,sitg,attributs/GE/temperature,CLIMAT_TEMPERATURE_14H00_P1_2020/CLIMAT_TEMPER...,NaN,NaN,NaN,NaN,NaN,NaN,opendata,attributs/GG/temperature,LST_GrandGeneve_2023.tif,NaN,0.25,0.5,100,250,defavorable,point,raster,temperature,point,raster,temperature,40.0,40.0,keep_nan,filtered,1,2056,"parquet, csv, gpkg",TB,True,True
9,Commodité,conflit_md,conflit_md,True,True,sitg,attributs/GE/network_couche_OCT.shp,RP_final_25112025.shp,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.25,0.5,1,1,defavorable,line,mean,Partage_us_score,point,sum,severity_score,72.0,2.0,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",TB,True,True


In [5]:
import fiona

gpkg_path = f"{input_file_path}/attributs/GG/osm/osm_attributes.gpkg"
print(fiona.listlayers(gpkg_path))

['borne_reparation', 'stationnement_velo', 'location', 'service_velo', 'fontaine', 'banc', 'toilette', 'amenite', 'crossing', 'traffic_signals', 'barrier', 'traffic_calming_point', 'transport_public', 'piste', 'bande', 'vitesse', 'chemin', 'revetement', 'etat_chaussee', 'eclairage', 'largeur', 'pente', 'giratoire', 'lac_cours_deau', 'canopee', 'espaces_ouverts', 'landuse_env', 'conflit_md']


**Attribut Accidents**

In [6]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----

attribute = 'accident'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
print(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
#gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
gdf = gpd.read_file(
    f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}",
    encoding="latin1"   # or "ISO-8859-1"
)
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------

gdf['filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)



# 
# # Initialize
# attribute = None
# row = None
# gdf = None
# print("Data initialized")

# # Load
# ###----change here-----
# attribute = 'accident'
# ###--------------------

# if territory == 'GE':

#     row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    
#     # Check availability
#     if row[attribute_source_path] == 'unavailable':
#         print(f"{attribute} non disponible pour le territoire {territory}")
#     else:
#         gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
#         gdf = gdf.to_crs(target_crs)

#         gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
#         gdf = gdf[gdf.geometry.notna()].copy()
#         print(f"Processing attribute: {attribute}")

#         # Filter
#         gdf['filtered'] = 0

#         ###----change here----------------------
#         gdf.loc[gdf.ANNEE > 2020, 'filtered'] = 1
#         gdf.loc[gdf.CONSEQ.isin(['Avec blessés graves', 'Avec tués']), 'filtered'] = 1
#         gdf.loc[gdf.VELOS == 1, 'filtered'] = 1
#         gdf.loc[gdf.VAE_25 == 1, 'filtered'] = 1
#         gdf.loc[gdf.VAE_45 == 1, 'filtered'] = 1
#         gdf.loc[gdf.PIETONS == 1, 'filtered'] = 1
#         ###--------------------------------------
#         print("Filters applied")

#         print(f"Proportion of features {attribute} kept after filtering:")
#         print(gdf.filtered.value_counts(normalize=True))

#         save(save_filtered_attributes, row, gdf, attribute)

# if territory == 'GG':
#     row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    
#     # Check availability
#     if row[attribute_source_path] == 'unavailable':
#         print(f"{attribute} non disponible pour le territoire {territory}")
#     else:
#         gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
#         gdf = gdf.to_crs(target_crs)

#         gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
#         gdf = gdf[gdf.geometry.notna()].copy()
#         print(f"Processing attribute: {attribute}")

#         # Filter
#         gdf['filtered'] = 0

#         ###----change here----------------------
#         gdf['filtered'] = 1

#         ###--------------------------------------
#         print("Filters applied")

#         print(f"Proportion of features {attribute} kept after filtering:")
#         print(gdf.filtered.value_counts(normalize=True))

#         save(save_filtered_attributes, row, gdf, attribute)

Data initialized
../../Data/input/attributs/GG/Merged_Accidents_2019_2023/Merged_GG_2019_2023.shp
Processing attribute: accident
Filters applied
Proportion of features accident kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: accident in format: parquet
Filtered data saved for attribute: accident in format: csv
Filtered data saved for attribute: accident in format: gpkg


**Attribut Vitesse**

In [7]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'vitesse'
###--------------------

if territory == 'GE':

    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:   
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf.loc[gdf.TYPE_ZONE == 0, 'filtered'] = 1


        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':

    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
        
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        
        gdf['filtered'] = 1

        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)


Data initialized
Processing attribute: vitesse
Filters applied
Proportion of features vitesse kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: vitesse in format: parquet
Filtered data saved for attribute: vitesse in format: csv
Filtered data saved for attribute: vitesse in format: gpkg


**Attribut Zone pietonne**

In [8]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'zone_pietonne'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:

        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf.loc[gdf.TYPE_ZONE.isin(['Zone piétonne', 'Zone de rencontre (20)', ]), 'filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:

        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf.loc[(gdf.speed_kph > 5) & (gdf.speed_kph < 30), 'filtered'] = 1



        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: zone_pietonne
Filters applied
Proportion of features zone_pietonne kept after filtering:
filtered
0    0.732541
1    0.267459
Name: proportion, dtype: float64
Filtered data saved for attribute: zone_pietonne in format: parquet
Filtered data saved for attribute: zone_pietonne in format: csv
Filtered data saved for attribute: zone_pietonne in format: gpkg


**Attribut Zone apaisée** 

In [9]:

# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'zone_apaisee'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")


        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf.loc[gdf.TYPE_LIMIT.isin(['Zone 30 km/h', 'Prescription 30 km/h', 'Prescription 20 km/h']), 'filtered'] = 1 

        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':

    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]

    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:

        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf.loc[(gdf.speed_kph > 5) & (gdf.speed_kph < 30), 'filtered'] = 1



        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: zone_apaisee
Filters applied
Proportion of features zone_apaisee kept after filtering:
filtered
0    0.732541
1    0.267459
Name: proportion, dtype: float64
Filtered data saved for attribute: zone_apaisee in format: parquet
Filtered data saved for attribute: zone_apaisee in format: csv
Filtered data saved for attribute: zone_apaisee in format: gpkg


**Lac et cours d'eau**

In [10]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'lac_cours_deau'
###--------------------
if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")

    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf.loc[gdf.GENRE.isin(["eau stagnante", "cours d'eau"]), 'filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)


Data initialized
Processing attribute: lac_cours_deau
Filters applied
Proportion of features lac_cours_deau kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: lac_cours_deau in format: parquet
Filtered data saved for attribute: lac_cours_deau in format: csv
Filtered data saved for attribute: lac_cours_deau in format: gpkg


**Fontaines**

In [11]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'fontaine'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: fontaine
Filters applied
Proportion of features fontaine kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: fontaine in format: parquet
Filtered data saved for attribute: fontaine in format: csv
Filtered data saved for attribute: fontaine in format: gpkg


**Rez Actifs**

In [12]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'rez_actif'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0


        # Filter criterion
        ###----change here----------------------
        keywords = ['commerce de détail', 'droguerie', 'Hôtels', 'Paroisses et associations religieuses', 'Petits commerces', 'Petits supermarchés', 'Restaurants', 'Salons', 'Écoles']

        rez_actif_keywords = [
            # Commerce de détail (générique)
            "commerce de détail",  # capte toutes les branches "commerce de détail ..."
            "grands magasins",
            "hypermarchés",
            "grands supermarchés", "petits supermarchés",
            "grands commerces", "petits commerces",
            "autres commerces de détail en magasin spécialisé",
            "autres commerces de détail en magasin non spécialisé",
            # "autres commerces de détail hors magasin, éventaires ou marchés",
            "commerce de détail alimentaire sur éventaires et marchés",

            # Hotel-restauration
            "restaurants, cafés, snack-bar",  # ligne exacte dans ta liste
            "restaurants avec possibilité d'hébergement",
            "services des traiteurs",
            "autres services de restauration",
            "bars",
            "boulangeries – tea-rooms",
            "salons de dégustation",
            "hôtels, auberges et pensions avec restaurant",
            "hôtels, auberges et pensions sans restaurant",
            # "hébergement collectif",
            # "autres hébergements",

            # Services de proximité et bien-être
            "salons de coiffure",
            "instituts de beauté",
            "parfumeries",
            "autres activités visant au bien-être physique",
            "centres de gymnastique et de fitness",
            "studios photographiques",
            "blanchisserie",
            "nettoyage à sec",
            "réparation de chaussures",
            "réparation d'autres biens personnels et domestiques",
            "réparation d'articles d'horlogerie et de bijouterie",
            "réparation de meubles et d'équipements du foyer",

            # Banques et assurances avec guichet
            "banques cantonales",
            "banques commerciales",
            "banques raiffeisen",
            "grandes banques",
            "autres banques",
            "succursales de banques étrangères",
            # "banquiers privés",
            # "établissements spécialisés dans le prêt personnel",
            # "caisses maladie",
            # "caisses de compensation",
            # "assurances contre les accidents et les dommages",
            # "assurance vie",
            # "autres assurances (sans la sécurité sociale obligatoire)",

            # Agences en contact avec le public
            "agences immobilières",
            "activités des marchands de biens immobiliers",
            "promotion immobilière",
            "activités des agences de voyage",
            "voyagistes; tour operator",
            "agences de placement de main-d'oeuvre",
            "agences de travail temporaire",
            # je n’inclus plus : centres d'appels, portails internet, agences de recouvrement (souvent en étage / back-office)

            # Culture / loisirs à rez de chaussée
            "gestion de salles de spectacles",
            "salles de spectacles",  # si présent
            "projection de films cinématographiques; cinémas",
            "cinémas",
            "discothèques, dancings, night clubs",
            "discothèques", "night clubs",
            "organisation de jeux de hasard et d'argent",

            # Funéraire
            "services funéraires",
        ]

        def flag_with_keywords(series, keywords):
            s = series.fillna("").str.lower()
            return s.apply(lambda v: any(kw.lower() in v for kw in keywords))

        gdf["filtered"] = flag_with_keywords(gdf["BRANCHE"], rez_actif_keywords).astype(int)


        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]

        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")

    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")
        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        rez_actif_set = {
        "restaurant","cafe","bar","pub","fast_food","ice_cream","biergarten",
        "bank","atm","bureau_de_change",
        "cinema","theatre","nightclub","casino",
        "marketplace","vending_machine"
        }

        gdf.loc[gdf["amenity"].isin(rez_actif_set), "filtered"] = 1

        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)



Data initialized
Processing attribute: rez_actif
Filters applied
Proportion of features rez_actif kept after filtering:
filtered
0    0.858004
1    0.141996
Name: proportion, dtype: float64
Filtered data saved for attribute: rez_actif in format: parquet
Filtered data saved for attribute: rez_actif in format: csv
Filtered data saved for attribute: rez_actif in format: gpkg


**Attribut Bruit**

In [13]:
attribute_source_path

'source_path_GG'

In [14]:

# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'bruit'
###--------------------
if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")


        # # Simplify geometries
        # simplify_tolerance = 0.01  # in meters
        # print(f"Simplifying geometries with tolerance = {simplify_tolerance} m ...")
        # gdf["geometry"] = gdf.geometry.simplify(simplify_tolerance, preserve_topology=True)
        # print("Simplification done.")
        # ###--------------------

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")


        # # Simplify geometries
        # simplify_tolerance = 0.01  # in meters
        # print(f"Simplifying geometries with tolerance = {simplify_tolerance} m ...")
        # gdf["geometry"] = gdf.geometry.simplify(simplify_tolerance, preserve_topology=True)
        # print("Simplification done.")
        # ###--------------------

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

Data initialized
bruit non disponible pour le territoire GG


**Attribut Proximité TP**



In [15]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'tp'
###--------------------

if territory == 'GE':

    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]

        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")

    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: tp
Filters applied
Proportion of features tp kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: tp in format: parquet
Filtered data saved for attribute: tp in format: csv
Filtered data saved for attribute: tp in format: gpkg


**Attribut Stationnement Genant**

In [16]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'stationnement_genant'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf[gdf.geometry.notna()].copy()  # Remove rows with None geometries
        gdf = gdf[gdf.geometry.is_valid].copy()  # Keep only valid geometries
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        ###--------------------------------------
        print("Filters applied")


        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf[gdf.geometry.notna()].copy()  # Remove rows with None geometries
        gdf = gdf[gdf.geometry.is_valid].copy()  # Keep only valid geometries
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        ###--------------------------------------
        print("Filters applied")


        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

Data initialized
stationnement_genant non disponible pour le territoire GG


**Attribut Proximité Aménités**

In [17]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'amenite'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        #keep if the column "BRANCHE" contains one of the following keywords (case insensitive)
        keywords = ['commerce de détail', 'droguerie', 'enseignement', 'Hôpitaux', 'Hôtels', "Maisons", 'Paroisses et associations religieuses', 'Petits commerces', 'Petits supermarchés', 'Protection civile', 'Restaurants', 'Salons', 'Écoles'  ]
        amenity_keywords = [
            # Santé et social
            "hôpital", "hôpitaux", "soins généraux",
            "hôpitaux spécialisés",
            "médecin", "médecins", "médecins généralistes",
            "dentair",  # dentaire, dentaires
            "laboratoires médicaux", "laboratoire médical",
            "paramédical", "paramédicales",
            "psychothérap", "psychologie",
            #"physiothérapie", "physio",
            #"infirmières", "soins à domicile",
            #"organisations pour la santé",
            #"institutions pour personnes handicapées",
            #"maisons pour personnes âgées",
            #"foyers pour enfants", "foyers pour enfants et adolescents",
            #"établissements pour les traitements psychosociaux",
            #"toxicomanes",
            #"action sociale",  # avec ou sans hébergement

            # Éducation
            "enseignement pré-primaire",
            "écoles primaires",
            "écoles obligatoires",
            "écoles de degré secondaire",
            "écoles de degré secondaire ii",
            "enseignement post-secondaire",
            "formation professionnelle de base",
            "formation professionnelle supérieure",
            "formation professionnelle",
            #"cours de langues",
            "enseignement culturel",
            "autres activités d'enseignements",
            "hautes écoles universitaires",
            "hautes écoles spécialisées",
            "hautes écoles pédagogiques",
            "écoles à programmes d'enseignement spécial",
            "cours de perfectionnement professionnel",

            # Culture, loisirs publics
            "gestion des bibliothèques", "bibliothèques", "archives",
            "gestion des musées",
            "gestion des sites et monuments historiques",
            "attractions touristiques similaires",
            "projection de films cinématographiques; cinémas",
            "cinémas",  # certaines lignes écrites différemment
            "orchestres, choeurs, musiciens",
            "troupes de théâtre et de ballet",
            "autres activités récréatives et de loisirs",
            "parcs d'attractions", "parcs à thèmes",
            "jardins botaniques", "jardins zoologiques", "réserves naturelles",

            # Sport
            "associations sportives", "piscine", "centre aquatique", "natation"
            "gestion d'installations sportives",
            "activités liées au sport",
            "enseignement de disciplines sportives",
            # (les centres de fitness peuvent être aussi rez actifs, on les met dans les deux)

            # Culte
            "paroisses et associations religieuses",
            "couvents et congrégations",

            # Services publics et sécurité
            #"administration publique générale",
            #"administration publique (tutelle)",
            #"poste dans le cadre d'une obligation de service universel",
            #"services du feu et de secours",
            #"activités d'ordre public et de sécurité",
            #"défense",
            "protection civile",
            "ambassades",
            "consulats",
            #"assurance-vieillesse et survivants",  # AVS, AI, AC
            #"assurance-invalidité",
            #"assurance-chômage",

            # Social élargi
            #"maisons d'éducation",
            #"foyers pour enfants et adolescents",
            #"appartements, maisons de vacances",  # si tu veux compter comme aménités touristiques
        ]



        def flag_with_keywords(series, keywords):
            s = series.fillna("").str.lower()
            return s.apply(lambda v: any(kw.lower() in v for kw in keywords))

        gdf["filtered"] = flag_with_keywords(gdf["BRANCHE"], amenity_keywords).astype(int)


        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")
        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        amenite_set = {
        "hospital","clinic","doctors","dentist","pharmacy",
        "school","kindergarten","college","university",
        "library","public_bookcase","arts_centre","museum",
        "place_of_worship",
        "police","fire_station","townhall","courthouse",
        "post_office",
        "community_centre","social_facility"
        }

        gdf.loc[gdf["amenity"].isin(amenite_set), "filtered"] = 1

        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: amenite
Filters applied
Proportion of features amenite kept after filtering:
filtered
0    0.941378
1    0.058622
Name: proportion, dtype: float64
Filtered data saved for attribute: amenite in format: parquet
Filtered data saved for attribute: amenite in format: csv
Filtered data saved for attribute: amenite in format: gpkg


**Attribut espaces ouverts**

In [18]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'espaces_ouverts'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)
        print(f"Processing attribute: {attribute}")


        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf.loc[gdf.TYPE == 'ESPACE PUBLIC', 'filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]

        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: espaces_ouverts
Filters applied
Proportion of features espaces_ouverts kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: espaces_ouverts in format: parquet
Filtered data saved for attribute: espaces_ouverts in format: csv
Filtered data saved for attribute: espaces_ouverts in format: gpkg


**Attribut Confort thermique**

In [19]:
# Initialize

attribute = 'temperature'
if territory == 'GE':
    
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:
        raster_path = (f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")

        with rasterio.open(raster_path) as src:
            # Read the raster data
            raster_data = src.read(1)  # Read first band
            
            # Define NoData
            nodata_val = src.nodata if src.nodata is not None else -9999
            
            # Valid mask: remove NoData
            valid_mask = (raster_data != nodata_val)
            
            # Get coordinates of valid points
            rows, cols = np.where(valid_mask)
            
            # Sample every 10th point
            rows, cols = rows[::10], cols[::10]
            
            # Get coordinates in map units
            xs, ys = rasterio.transform.xy(src.transform, rows, cols)
            temp_values = raster_data[rows, cols]
            
            # Create GeoDataFrame directly with filtered points
            gdf = gpd.GeoDataFrame({
                'temperature': temp_values,
                'filtered': 1,  # All points are filtered since we pre-filtered the data
                'geometry': [Point(x, y) for x, y in zip(xs, ys)]
            }, geometry='geometry')
            
            # Set CRS and convert to target CRS
            gdf = gdf.set_crs(src.crs)
            gdf = gdf.to_crs(target_crs)

        print(f"Processing attribute: {attribute}")
        print("\nDescriptive statistics:")
        print(gdf['temperature'].describe())

        # Save
        print(f"Saving {attribute}: can take up to 3mn")
        save(save_filtered_attributes, row, gdf, attribute)



if territory == 'GG':
    
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:
        raster_path = (f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")

        with rasterio.open(raster_path) as src:
            # Read the raster data
            raster_data = src.read(1)  # Read first band
            
            # Define NoData
            nodata_val = src.nodata if src.nodata is not None else -9999
            
            # Valid mask: remove NoData
            valid_mask = (raster_data != nodata_val)
            
            # Get coordinates of valid points
            rows, cols = np.where(valid_mask)
            
            # Sample every 10th point
            rows, cols = rows[::10], cols[::10]
            
            # Get coordinates in map units
            xs, ys = rasterio.transform.xy(src.transform, rows, cols)
            temp_values = raster_data[rows, cols]
            
            # Create GeoDataFrame directly with filtered points
            gdf = gpd.GeoDataFrame({
                'temperature': temp_values,
                'filtered': 1,  # All points are filtered since we pre-filtered the data
                'geometry': [Point(x, y) for x, y in zip(xs, ys)]
            }, geometry='geometry')
            
            # Set CRS and convert to target CRS
            gdf = gdf.set_crs(src.crs)
            gdf = gdf.to_crs(target_crs)

        print(f"Processing attribute: {attribute}")
        print("\nDescriptive statistics:")
        print(gdf['temperature'].describe())

        # Save
        print(f"Saving {attribute}: can take up to 3mn")
        save(save_filtered_attributes, row, gdf, attribute)


Processing attribute: temperature

Descriptive statistics:
count    643595.000000
mean         30.568608
std           6.033808
min           9.956015
25%          25.599438
50%          29.507089
75%          35.122896
max          58.245801
Name: temperature, dtype: float64
Saving temperature: can take up to 3mn
Filtered data saved for attribute: temperature in format: parquet
Filtered data saved for attribute: temperature in format: csv
Filtered data saved for attribute: temperature in format: gpkg


**Attribut Largeur trottoir**

In [20]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'largeur_trottoir'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]

    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")

    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)
        print(f"Processing attribute: {attribute}")


        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        gdf['Largeur_score'] = 0
        gdf['Largeur_score'] = gdf['Largeur'].map({'Très large': 5, 'Large': 4, 'Moyen': 3, 'Etroit': 2, 'Très étroit': 1, 'Pas de trottoir': 0})
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]

    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")

    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)
        print(f"Processing attribute: {attribute}")


        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        gdf['Largeur_score'] = 0
        gdf['Largeur_score'] = gdf['Largeur'].map({'Très large': 5, 'Large': 4, 'Moyen': 3, 'Etroit': 2, 'Très étroit': 1, 'Pas de trottoir': 0})
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

Data initialized
largeur_trottoir non disponible pour le territoire GG


**Attribut Chemin**

In [21]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'chemin'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]

    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")

    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf.loc[gdf.OBJET == "Chemin" , 'filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf.loc[gdf.highway=='path', 'filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: chemin
Filters applied
Proportion of features chemin kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: chemin in format: parquet
Filtered data saved for attribute: chemin in format: csv
Filtered data saved for attribute: chemin in format: gpkg


**Attribut conflit usages**

In [22]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'conflit_md'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")

    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")


        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        gdf['Partage_us_score'] = 0
        gdf['Partage_us_score'] = gdf['Partage_us'].map({'Trafic motorisé' : 4, 'Mixité vélos - Piste sur trotto':3, 'Mixité ayants droit motorisés':2, 'Mixité vélos':1, 'Mixité vélo':1})
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")

    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")


        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        gdf['severity_score'] = 0
        gdf['severity_score'] = gdf['severity'].map({'high' : 3, 'medium':2, 'low':1})
        
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: conflit_md
Filters applied
Proportion of features conflit_md kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: conflit_md in format: parquet
Filtered data saved for attribute: conflit_md in format: csv
Filtered data saved for attribute: conflit_md in format: gpkg


**Attribut Topographie**

In [23]:
## !!! Depreciated since june 2026, now computed in 1_1
# 
# # # Initialize
# attribute = None
# row = None
# gdf = None
# print("Data initialized")

# # Load
# ###----change here-----
# attribute = 'pente'
# ###--------------------

# if territory == 'GE':
#     row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
#         # Check availability
#     if row[attribute_source_path] == 'unavailable':
#         print(f"{attribute} non disponible pour le territoire {territory}")

#     else:
#         gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
#         gdf = gdf.to_crs(target_crs)
#         print(f"Processing attribute: {attribute}")


#         # Filter
#         # Initialize
#         gdf['filtered'] = 0

#         # Filter criterion
#         ###----change here----------------------
#         gdf['filtered'] = 1
#         ###--------------------------------------
#         print("Filters applied")

#         # Count and proportion
#         print(f"Proportion of features {attribute} kept after filtering:")
#         print(gdf.filtered.value_counts(normalize=True))

#         # Save
#         save(save_filtered_attributes, row, gdf, attribute)


# if territory == 'GG':
#     row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
#         # Check availability
#     if row[attribute_source_path] == 'unavailable':
#         print(f"{attribute} non disponible pour le territoire {territory}")

#     else:
#         raster_path = (f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")

#         with rasterio.open(raster_path) as src:
#             # Read the raster data
#             raster_data = src.read(1)  # Read first band
            
#             # Define NoData
#             nodata_val = src.nodata if src.nodata is not None else -9999
            
#             # Valid mask: remove NoData
#             valid_mask = (raster_data != nodata_val)
            
#             # Get coordinates of valid points
#             rows, cols = np.where(valid_mask)
            
#             # Sample every 10th point
#             rows, cols = rows[::10], cols[::10]
            
#             # Get coordinates in map units
#             xs, ys = rasterio.transform.xy(src.transform, rows, cols)
#             temp_values = raster_data[rows, cols]
            
#             # Create GeoDataFrame directly with filtered points
#             gdf = gpd.GeoDataFrame({
#                 'slope_degree': temp_values,
#                 'filtered': 1,  # All points are filtered since we pre-filtered the data
#                 'geometry': [Point(x, y) for x, y in zip(xs, ys)]
#             }, geometry='geometry')
            
#             # Set CRS and convert to target CRS
#             gdf = gdf.set_crs(src.crs)
#             gdf = gdf.to_crs(target_crs)

#         print(f"Processing attribute: {attribute}")
#         print("\nDescriptive statistics:")
#         print(gdf['slope_degree'].describe())

#         # Save
#         print(f"Saving {attribute}: can take up to 3mn")
# save(save_filtered_attributes, row, gdf, attribute)

**Attribut Canopée**

In [24]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'canopee'
###--------------------
if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")


        # Simplify geometries
        simplify_tolerance = 0.01  # in meters
        print(f"Simplifying geometries with tolerance = {simplify_tolerance} m ...")
        gdf["geometry"] = gdf.geometry.simplify(simplify_tolerance, preserve_topology=True)
        print("Simplification done.")
        ###--------------------

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")

    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1

        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: canopee
Filters applied
Proportion of features canopee kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: canopee in format: parquet
Filtered data saved for attribute: canopee in format: csv
Filtered data saved for attribute: canopee in format: gpkg


** Attribut Envrionnement Naturel Ouvert

In [25]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'environnement_ouvert_naturel'
###--------------------
if territory == 'GE':
    print(f"{attribute} : pas de routine pour GE (vectorisation par carreaux implémentée pour GG uniquement)")

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")

    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1

        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: environnement_ouvert_naturel
Filters applied
Proportion of features environnement_ouvert_naturel kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: environnement_ouvert_naturel in format: parquet
Filtered data saved for attribute: environnement_ouvert_naturel in format: csv
Filtered data saved for attribute: environnement_ouvert_naturel in format: gpkg


**Attributs Bancs**

In [26]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'banc'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")


        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
        
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")


        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: banc
Filters applied
Proportion of features banc kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: banc in format: parquet
Filtered data saved for attribute: banc in format: csv
Filtered data saved for attribute: banc in format: gpkg


**Attribut Toilette**

In [27]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'toilette'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")

    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")


        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")

    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")


        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: toilette
Filters applied
Proportion of features toilette kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: toilette in format: parquet
Filtered data saved for attribute: toilette in format: csv
Filtered data saved for attribute: toilette in format: gpkg


**Attribut eclairage**

In [28]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'eclairage'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")


        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")


        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

Data initialized
Processing attribute: eclairage
Filters applied
Proportion of features eclairage kept after filtering:
filtered
1    1.0
Name: proportion, dtype: float64
Filtered data saved for attribute: eclairage in format: parquet
Filtered data saved for attribute: eclairage in format: csv
Filtered data saved for attribute: eclairage in format: gpkg


**Attribut air**

In [29]:
# Initialize

attribute = 'air'
if territory == 'GE':
    
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    
    else:
        raster_path = (f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")

        with rasterio.open(raster_path) as src:
            # Read the raster data
            raster_data = src.read(1)  # Read first band
            
            # Define NoData
            nodata_val = src.nodata if src.nodata is not None else -9999
            
            # Valid mask: remove NoData
            valid_mask = (raster_data != nodata_val)
            
            # Get coordinates of valid points
            rows, cols = np.where(valid_mask)
            
            # Sample every 10th point
            rows, cols = rows[::1], cols[::1]
            
            # Get coordinates in map units
            xs, ys = rasterio.transform.xy(src.transform, rows, cols)
            temp_values = raster_data[rows, cols]
            
            # ✅ Conversion en µmol/m² pour éviter les underflows d'affichage
            temp_values_converted = temp_values * 1e6  # mol/m² → µmol/m²
            
            gdf = gpd.GeoDataFrame({
                'no_2': temp_values_converted,   # valeurs lisibles ex: 18 à 40 µmol/m²
                'no_2_raw': temp_values,         # valeurs originales conservées si besoin
                'filtered': 1,
                'geometry': [Point(x, y) for x, y in zip(xs, ys)]
            }, geometry='geometry')
            
            gdf = gdf.set_crs(src.crs)
            gdf = gdf.to_crs(target_crs)

        print(f"Processing attribute: {attribute}")
        print("\nDescriptive statistics:")
        print(gdf['no_2'].describe())

        # Save
        print(f"Saving {attribute}: can take up to 3mn")
        save(save_filtered_attributes, row, gdf, attribute)


if territory == 'GG':
    
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    
    else:
        raster_path = (f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")

        with rasterio.open(raster_path) as src:
            # Read the raster data
            raster_data = src.read(1)  # Read first band
            
            # Define NoData
            nodata_val = src.nodata if src.nodata is not None else -9999
            
            # Valid mask: remove NoData
            valid_mask = (raster_data != nodata_val)
            
            # Get coordinates of valid points
            rows, cols = np.where(valid_mask)
            
            # Sample every 10th point
            rows, cols = rows[::10], cols[::10]
            
            # Get coordinates in map units
            xs, ys = rasterio.transform.xy(src.transform, rows, cols)
            temp_values = raster_data[rows, cols]
            
            # ✅ Conversion en µmol/m² pour éviter les underflows d'affichage
            temp_values_converted = temp_values * 1e6  # mol/m² → µmol/m²
            
            gdf = gpd.GeoDataFrame({
                'no_2': temp_values_converted,   # valeurs lisibles ex: 18 à 40 µmol/m²
                'no_2_raw': temp_values,         # valeurs originales conservées si besoin
                'filtered': 1,
                'geometry': [Point(x, y) for x, y in zip(xs, ys)]
            }, geometry='geometry')
            
            gdf = gdf.set_crs(src.crs)
            gdf = gdf.to_crs(target_crs)

        print(f"Processing attribute: {attribute}")
        print("\nDescriptive statistics:")
        print(gdf['no_2'].describe())

        # Save
        print(f"Saving {attribute}: can take up to 3mn")
        save(save_filtered_attributes, row, gdf, attribute)

Processing attribute: air

Descriptive statistics:
count    536.000000
mean      29.936349
std        3.170093
min       20.915861
25%       27.628236
50%       29.891173
75%       31.946087
max       40.489754
Name: no_2, dtype: float64
Saving air: can take up to 3mn
Filtered data saved for attribute: air in format: parquet
Filtered data saved for attribute: air in format: csv
Filtered data saved for attribute: air in format: gpkg


**Attribut Charge trafic**

Raster 10 m (déjà lissé). Seuls les carreaux touchés par le réseau (masque dilaté d'un pixel) sont vectorisés en carrés, pour un calcul ultérieur de moyenne pondérée par la longueur traversée. Implémenté pour GG uniquement.

In [30]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'charge'
value_column_trafic = 'charge'   # nom de la colonne valeur -> à reporter dans value_column_GG de l'Excel
DILATION_PIXELS = 1                     # marge de sécurité autour du réseau (en pixels)
###--------------------

if territory == 'GE':
    print(f"{attribute} : pas de routine pour GE (vectorisation par carreaux implémentée pour GG uniquement)")

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:
        raster_path = f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}"
        print(f"Processing attribute: {attribute}")

        # Vectorisation des seuls carreaux touchés par le réseau (segmented_net chargé plus haut)
        gdf = raster_cells_touching_network(
            raster_path,
            segmented_net,
            value_column=value_column_trafic,
            dilation_pixels=DILATION_PIXELS,
            keep_nodata=False,
        )

        # Filter
        ###----change here----------------------
        gdf['filtered'] = 1
        ###--------------------------------------
        
        SCALE = 1e6
        gdf[f"{value_column_trafic}_raw"] = gdf[value_column_trafic]   # valeur d'origine conservée
        gdf[value_column_trafic] = gdf[value_column_trafic] * SCALE
        print("Filters applied")

        # NB : volontairement PAS de to_crs(target_crs) ici.
        # Les carrés restent dans le CRS du raster (EPSG:2056) pour rester exacts ;
        # le 2_2 reprojette de toute façon vers le CRS des segments si besoin.

        print("\nDescriptive statistics:")
        print(gdf[value_column_trafic].describe())

        # Save
        save(save_filtered_attributes, row, gdf, attribute)


Data initialized
Processing attribute: charge
  Fenêtre lue : 6891 x 6553 pixels (raster complet : 6891 x 6553)
  Pixels touchés par le réseau : 1,784,942
  Après dilatation (1 px) : 4,728,117 carreaux vectorisés
  ⚠️ 26,967 pixels touchés par le réseau sont NoData (exclus)
Filters applied

Descriptive statistics:
count    4.728117e+06
mean     2.086716e+02
std      7.098758e+02
min      0.000000e+00
25%      0.000000e+00
50%      2.962727e+00
75%      6.925361e+01
max      2.252017e+04
Name: charge, dtype: float64
Filtered data saved for attribute: charge in format: parquet
Filtered data saved for attribute: charge in format: csv
Filtered data saved for attribute: charge in format: gpkg
